# 4 — Report PDF

Wraps **every results figure into one multi-page PDF** — the shareable deliverable. It reads only the
saved tables from notebook 1 (no logs, no video) and calls the same plotting code the other notebooks
use, so the PDF and the notebooks never diverge.

Sections: 1 dataset overview · 2 each task per animal + across animals · 3 per-session pattern +
across-mice average · 4 first vs second half · 5 path clusters. A section that cannot be drawn (e.g.
a task with no clustered trials) becomes a short note page instead of failing the whole report.

## Load + build

In [ ]:
MAIN_DIR = '/mnt/server/data'
PIPELINE_DIR = None
OUT_PDF = None            # None -> saved next to the dataset as results_report.pdf
# =============================================================================
import sys, importlib
from pathlib import Path
import matplotlib.pyplot as plt

cands = ([Path(PIPELINE_DIR)] if PIPELINE_DIR else []) + [
    Path.cwd().parent, Path.cwd(), Path.cwd().parent / 'session_pipeline']
PIPE = next((c.resolve() for c in cands if (c / 'common' / 'session_index.py').exists()), None)
sys.path.insert(0, str(PIPE / 'common')); sys.path.insert(0, str(PIPE / 'results'))

import build_log_df as bl, make_report as mr
bl = importlib.reload(bl); mr = importlib.reload(mr)

_LOCAL = Path('~/repo/session_pipeline_output').expanduser()
OUT = _LOCAL if (_LOCAL / 'df_sessions.pkl').exists() else (Path(MAIN_DIR).expanduser() / 'df_log')
df_sessions = bl.load(OUT / 'df_sessions.pkl')
df_trials   = bl.load(OUT / 'df_trials.pkl')

out_pdf = Path(OUT_PDF) if OUT_PDF else (OUT / 'results_report.pdf')
mr.build_pdf(df_sessions, df_trials, out_pdf)
print('done ->', out_pdf)

## Preview (optional)

Renders the pages inline so you can check the PDF without leaving the notebook.

In [ ]:
try:
    from pdf2image import convert_from_path
    for i, img in enumerate(convert_from_path(str(out_pdf), dpi=90)):
        plt.figure(figsize=(11, 8.5)); plt.imshow(img); plt.axis('off'); plt.title(f'page {i+1}')
        plt.show()
except Exception as e:
    print('inline preview needs pdf2image + poppler; open the PDF directly instead:', e)
    print(out_pdf)